<a href="https://colab.research.google.com/github/ankitroy1st456-lang/Ai-pronunciation-coach/blob/main/pytorch-lstm-next-word-predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#pytorch-lstm-next-word-predictor



In [1]:
!pip install nltk

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset,DataLoader
import nltk


In [3]:
!unzip /content/archive.zip -d /content

Archive:  /content/archive.zip
  inflating: /content/next_word_predictor.txt  


In [206]:
with open("/content/next_word_predictor.txt","r") as file:
  document = file.read()
  print(len(document))

167445


In [197]:
# Tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [207]:
length = len(document)//2
length

83722

In [208]:
document= document[:length]

In [259]:
print(document[:100])

The sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the ta


In [209]:
# tokenize
tokens = nltk.word_tokenize(document.lower())

In [210]:
# build vocab

vocab = {'<unknown>':0}
for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token]=len(vocab)

In [211]:
len(vocab)

3490

In [212]:
# find out sentences - extract sentences
input_sentences = document.split('.')


In [213]:
input_sentences[0]

'The sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the tall trees'

In [214]:
def text_to_indices(sentence,vocab):
  numerical_sentence=[]
  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unknown>'])

  return numerical_sentence

In [215]:
input_numerical_sentences=[]
for sentence in input_sentences:
  input_numerical_sentences.append(text_to_indices(nltk.word_tokenize(sentence.lower()),vocab))


In [216]:
len(input_numerical_sentences)

956

In [217]:
input_numerical_sentences[0]

[1, 2, 3, 4, 5, 6, 1, 7, 8, 9, 10, 11, 12, 13, 14, 15, 1, 16, 17, 1, 18, 19]

In [219]:
training_sequence = []
for sentence in input_numerical_sentences:
  for i in range(1,len(sentence)):
    training_sequence.append(sentence[:i+1])



In [220]:
len(training_sequence)

14724

In [221]:
# padding sentences - adding pre zeros '0'

# find longest seq first :
length_list=[]
for seq in training_sequence:
    length_list.append(len(seq))

In [222]:
max(length_list)

85

In [223]:
for sequence in training_sequence:
  print(sequence)
  break

[1, 2]


In [224]:
padded_training_sequence=[]
for sequence in training_sequence:
  padded_training_sequence.append([0]*(max(length_list)-len(sequence)) + sequence)


In [225]:
padded_training_sequence = torch.tensor(padded_training_sequence,dtype=torch.long)

In [226]:
padded_training_sequence.shape

torch.Size([14724, 85])

In [227]:
X = padded_training_sequence[:,:-1]
y = padded_training_sequence[:,-1]

In [228]:
y

tensor([   2,    3,    4,  ..., 2316,   69, 2021])

In [229]:
class CustomDataset(Dataset):
  def __init__(self,X,y):
    self.X=X
    self.y=y

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx],self.y[idx]

In [230]:
dataset = CustomDataset(X,y)

In [231]:
len(dataset)

14724

In [232]:
dataloader = DataLoader(dataset, batch_size = 32, shuffle=True)

In [233]:
for input,output in dataloader:
  print(input,output)
  break

tensor([[   0,    0,    0,  ...,    0,    0,  676],
        [   0,    0,    0,  ...,   10,   11, 1578],
        [   0,    0,    0,  ..., 1535,   22, 2766],
        ...,
        [   0,    0,    0,  ...,    0, 1625, 1511],
        [   0,    0,    0,  ...,    0,    0, 3079],
        [   0,    0,    0,  ...,    1, 2519, 2520]]) tensor([2979,   17,  596, 1498,  300, 1130, 1925,   10, 2483,  300,   12,    3,
         241, 1613,   11,   45, 1358, 1196,  400,   10, 1130,   49, 2179,    1,
          10, 2100,   10, 2680,  914, 1199, 1130,  104])


In [234]:
class LSTMmodel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size,100)
    self.lstm = nn.LSTM(100,150,batch_first=True)
    self.fc = nn.Linear(150,vocab_size)
    pass

  def forward(self,x):
    embedded = self.embedding(x)
    intermediate_hidden_state, (final_hidden_state,final_cell_state)=self.lstm(embedded)
    output = self.fc(final_hidden_state.squeeze(0))
    return output


In [235]:
x = nn.Embedding(3490,100)
y = nn.LSTM(100,150,batch_first=True)
z = nn.Linear(150,3490)

In [236]:
a=dataset[0][0].unsqueeze(0)

In [237]:
b=x(a)

In [238]:
c,d=y(b)

In [239]:
c.shape

torch.Size([1, 84, 150])

In [240]:
e,f=d

In [241]:
z(f.squeeze(0)).shape

torch.Size([1, 3490])

In [252]:
model = LSTMmodel(len(vocab))

In [253]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [254]:
model.to(device)

LSTMmodel(
  (embedding): Embedding(3490, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=3490, bias=True)
)

In [255]:
epochs = 20
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [256]:
#training loop

for epoch in range(epochs):
  total_loss = 0
  for batch_x,batch_y in dataloader:
    batch_x,batch_y = batch_x.to(device),batch_y.to(device)

    optimizer.zero_grad()

    output = model(batch_x)

    loss = criterion(output,batch_y)

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  total_loss = total_loss/len(dataloader)
  print(f"Epoch: {epoch+1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 6.7596
Epoch: 2, Loss: 5.7832
Epoch: 3, Loss: 5.0807
Epoch: 4, Loss: 4.4096
Epoch: 5, Loss: 3.7841
Epoch: 6, Loss: 3.2115
Epoch: 7, Loss: 2.6966
Epoch: 8, Loss: 2.2468
Epoch: 9, Loss: 1.8637
Epoch: 10, Loss: 1.5462
Epoch: 11, Loss: 1.2823
Epoch: 12, Loss: 1.0650
Epoch: 13, Loss: 0.8887
Epoch: 14, Loss: 0.7436
Epoch: 15, Loss: 0.6275
Epoch: 16, Loss: 0.5333
Epoch: 17, Loss: 0.4607
Epoch: 18, Loss: 0.4005
Epoch: 19, Loss: 0.3562
Epoch: 20, Loss: 0.3227


In [294]:
def prediction(model,vocab,text):
  # tokenize word
  tokenized_text = nltk.word_tokenize(text.lower())
  # print(tokenized_text)

  # text to numerical indices
  numerical_text = text_to_indices(tokenized_text,vocab)
  # print(numerical_text)

  # numerical indices -> pre zero padding
  padded_text = [0]*(85-len(numerical_text)) + numerical_text
  padded_text_tensor = torch.tensor(padded_text,dtype=torch.long).unsqueeze(0)
  # print(padded_text_tensor)

  # padded text -> model
  output = model(padded_text_tensor.to(device))
  # print(output)

  # output -> predicted word index
  value,index = torch.max(output,dim=1)
  # print(index)

  # index -> fetch word -> merge text to user sentence and display
  return list(vocab.keys())[index]



In [289]:
prediction(model,vocab,"The sun was shining")

'The sun was shining brightly'

In [301]:
num_tokens=10
data ="The sun was shining"
for i in range(num_tokens):
  output=prediction(model,vocab,data)

  data=data+" "+output

  print(output)


brightly
in
the
clear
blue
sky
,
and
a
gentle
